In [4]:
from langchain_community.retrievers import WikipediaRetriever

/var/folders/dz/m4wfm9b56t3_g12zg0y1tlk40000gn/T/ipykernel_92195/2879121110.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [15]:
query= "Shifting from deep learning to genai"

In [16]:
retriver = WikipediaRetriever(top_k_results=3, lang='en')
answers = retriver.invoke(query)

for index, doc in enumerate(answers):
    print(f"\nResult {index+ 1}", doc.page_content)


Result 1 Generative artificial intelligence (GenAI) is a subfield of artificial intelligence (AI) that uses generative models to generate text, images, videos, audio, computer code or other forms of digital data. These models learn the underlying patterns and structures of their training data, and use them to generate new data in response to input, which often takes the form of natural language prompts.
The prevalence of generative AI tools has increased significantly since an AI boom beginning in the early 2020s. This boom was made possible by improvements in deep neural networks, particularly large language models (LLMs), which are based on the transformer architecture. Generative AI applications include chatbots such as ChatGPT, Claude, Microsoft Copilot, DeepSeek, Doubao, Google Gemini, Grok, Kimi and Qwen; text-to-image models such as DALL-E, Firefly, Stable Diffusion, and Midjourney; and text-to-video models such as Veo, LTX and Sora.
Companies in a variety of sectors have used 

In [ ]:
#vector store retriever

In [58]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [59]:
# need to install the model as well, dimensions are default 384
embeddings = HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15639.06it/s]


In [73]:
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
    id=2,
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
    id=3,
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
    id=4,
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
    id=5,
)
document_6 = Document(
    page_content="Building an exciting new project with LangChain - lets check it out!",
    metadata={"source": "tweet"},
    id=6,
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6
]

In [63]:
vector_store = Chroma.from_documents(embedding=embeddings, documents=documents,
                      persist_directory='chroma_retriver', collection_name='retriver_collection')

In [64]:
retriever = vector_store.as_retriever(search_kwargs= {"k" : 3})

In [65]:
results = retriever.invoke("LangChain provides abstractions to make working with LLMs easy")

for index, doc in enumerate(results):
    print(f"\n Result{index + 1}", doc.page_content)

result = []


 Result1 Building an exciting new project with LangChain - come check it out!

 Result2 Robbers broke into the city bank and stole $1 million in cash.

 Result3 I had chocolate chip pancakes and scrambled eggs for breakfast this morning.


In [66]:
# MMR

In [74]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# need to install the model as well, dimensions are default 384
embeddings = HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9276.84it/s]


In [75]:
vector_store = FAISS.from_documents(documents=documents, embedding= embeddings)


In [82]:
retriever = vector_store.as_retriever(search_type = "mmr", search_kwargs={"k": 3, "lambda_mult": 0.5}, )
results = retriever.invoke(
    "LangChain provides abstractions to make working with LLMs easy")

results
# result normally: 6, 3, 4
# with mmr: 6, 4, 2
# lambda_mult = 1 means how much diversification allowed 

[Document(id='6', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - lets check it out!'),
 Document(id='4', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(id='2', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')]